# ML-10 — Content Action Playbook (Lane 2)
[Colab](https://colab.research.google.com/github/himanshu-yadav-10/Flyrank-ML-starter-template/blob/main/work/notebooks/w07_action_playbook.ipynb)
The ranked action engine produced by the capstone model — what to do first, and why, in words a human trusts.

In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib
import os, sys, subprocess
from pathlib import Path
import numpy as np, pandas as pd

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.isdir("Flyrank-ML-starter-template"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/himanshu-yadav-10/Flyrank-ML-starter-template",
                        "Flyrank-ML-starter-template"], check=True)
    os.chdir("Flyrank-ML-starter-template")
    REPO = Path(os.getcwd())
else:
    here = Path(os.getcwd()).resolve()
    REPO = next((p for p in [here, *here.parents] if (p / "data" / "raw" / "content_refresh_anonymized.csv").exists()), None)

assert REPO is not None, "repo root (with data/raw/) not found"
sys.path.insert(0, str(REPO / "scripts"))
os.chdir(REPO)
print("Repo:", REPO)



[notice] A new release of pip is available: 26.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


Repo: C:\Users\Himanshu Yadav\Desktop\Flyrank\Flyrank-ML-starter-template


## 1. Ranked actions + reason codes
Working the queue top-down, the model's reason codes map to concrete actions. Order is by
decline probability (break ties by value/effort).

In [2]:
import pandas as pd, numpy as np
from pathlib import Path
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/"data"/"raw"/"content_refresh_anonymized.csv").exists()), None)
q = pd.read_csv(REPO/"work"/"outputs"/"capstone_opportunity_queue.csv")
print("Queue loaded:", len(q), "held-out pages")
print()
print(q.groupby(["action","reason_code"]).size().to_string())
print()
print("Top 5 by opportunity score:")
cols=["opportunity_rank","opportunity_score","reason_code","action","impressions_90d","avg_position","ctr","days_since_last_update"]
print(q[cols].head(5).to_string(index=False))


Queue loaded: 600 held-out pages

action               reason_code           
metadata_review      visible_low_ctr           304
refresh_or_rewrite   mature_content_decline     19
review               position_loss              68
                     reach_loss                 70
                     review_decline_risk        64
review_low_priority  low_reach_review           75

Top 5 by opportunity score:
 opportunity_rank  opportunity_score     reason_code          action  impressions_90d  avg_position  ctr  days_since_last_update
                1           0.882366 visible_low_ctr metadata_review              767           3.0 0.00                       8
                2           0.877300 visible_low_ctr metadata_review            12053           2.6 0.02                       8
                3           0.864381   position_loss          review               21          45.0 0.00                       1
                4           0.854881 visible_low_ctr metadata_review   

In [3]:
# Discretised action playbook ordered by priority (decisional, decision-support only)
playbook = pd.DataFrame([
    ("1. refresh",            "stale_visible_decline",      "stale + high-traction page at decline risk: strongest refresh candidate"),
    ("2. refresh_or_rewrite", "mature_content_decline",     "ageing content trending down: refresh or rewrite"),
    ("3. metadata_review",    "visible_low_ctr",            "still earning impressions but under-converting: fix title/meta/CTR"),
    ("4. review",             "reach_loss / position_loss / review_decline_risk", "decline probability high - inspect before editing"),
    ("5. review_low_priority","low_reach_review",           "few impressions: worth a look only if high-value/strategic"),
    ("6. monitor",            "monitor_borderline / monitor_stable", "not at clear risk: leave alone, watch"),
], columns=["priority","reason_code","why"])
print(playbook.to_string(index=False))


              priority                                      reason_code                                                                     why
            1. refresh                            stale_visible_decline stale + high-traction page at decline risk: strongest refresh candidate
 2. refresh_or_rewrite                           mature_content_decline                        ageing content trending down: refresh or rewrite
    3. metadata_review                                  visible_low_ctr      still earning impressions but under-converting: fix title/meta/CTR
             4. review reach_loss / position_loss / review_decline_risk                       decline probability high - inspect before editing
5. review_low_priority                                 low_reach_review              few impressions: worth a look only if high-value/strategic
            6. monitor              monitor_borderline / monitor_stable                                   not at clear risk: leave alone

## 2. Intended use and limits
**Who:** the FlyRank content/review team. **For what:** ordering the daily/weekly review
backlog — which pages to refresh, rewrite, fix metadata on, or monitor. **Limits:** this is
decision-support; it orders *observed decline risk*, it does not measure the payoff of any
action, and it is built on the starter 30k slice, one 90-day window, six held-out clients.

In [4]:
print("Use: rank only. The action labels are heuristics tied to observed feature profiles,")
print("not guaranteed outcomes. Re-validate on new data before acting at scale.")


Use: rank only. The action labels are heuristics tied to observed feature profiles,
not guaranteed outcomes. Re-validate on new data before acting at scale.


## 3. Human review + the no-go list
A person must confirm before acting: (1) is the page worth the writing effort? (2) is traffic
high enough to matter (volume floor)? (3) does the content actually warrant a refresh?
**Never auto-publish** rewrites, never act on a single-page signal alone, never edit purely
because a monitor-tier page ranks low.

In [5]:
print("No-go: no automated publishing, no single-signal edits, no content changes on monitor tier.")
print("Human gates: value/effort check, volume floor, editorial judgement.")


No-go: no automated publishing, no single-signal edits, no content changes on monitor tier.
Human gates: value/effort check, volume floor, editorial judgement.


## 4. Monitoring / retrain triggers
Refresh the queue when: a new data release lands, the label definition or feature set changes,
or the model's precision on a fresh holdout drifts below an agreed threshold (e.g. precision@50
drops materially). Watch distribution shift in the top drivers (reach stability, content age).

In [6]:
print("Retrain triggers: new release, feature/label change, or precision@50 drift on a fresh holdout.")
print("Monitor: distribution shift in days_with_impressions, content_age_days, avg_position.")


Retrain triggers: new release, feature/label change, or precision@50 drift on a fresh holdout.
Monitor: distribution shift in days_with_impressions, content_age_days, avg_position.


## 5. Exports for the paper
The queue and this playbook feed the paper's "Ranked recommendations" section; figures and
metrics feed the results section.

In [7]:
import json
meta = json.loads((REPO/"work"/"outputs"/"capstone_metrics.json").read_text())
print("Exported to work/outputs/: capstone_opportunity_queue.csv, capstone_metrics.json")
print("Figures in work/figures/: precision_at_k, pr_curve, roc_curve, feature_importance")
print("Paper: docs/index.html (all numbers reproduced here).")


Exported to work/outputs/: capstone_opportunity_queue.csv, capstone_metrics.json
Figures in work/figures/: precision_at_k, pr_curve, roc_curve, feature_importance
Paper: docs/index.html (all numbers reproduced here).


## Self-check
- [x] Sections filled with reasoning AND code
- [x] Runs top to bottom with outputs
- [x] No client names/URLs/private queries
- [x] Observed / measured / directional / decision-support language
- [x] Committed under work/notebooks/
